# Update the webapp with the latest `pipeline_cards.json`

Companion to the **`create_pipeline_cards`** pipeline, which writes
`pipeline_cards.json` (archiving any previous version into a `historical/`
subfolder) but **cannot deploy it** — a pipeline-scoped token always fails
`updateWebapp` with `PERMISSION_DENIED` (see this folder's `README.md`).

This notebook runs as **your user**, so it has the workspace EDITOR/ADMIN rights the
mutation requires. Run the cells top to bottom:

1. **Config** — set `WEBAPP_NAME` if you are targeting a different variant.
2. **Locate** — reads `pipeline_cards.json` and reports its age.
3. **Resolve + pre-flight** — finds the webapp by name and checks `permissions.update`.
4. **Deploy** — sends only `pipeline_cards.json`.

Any problem raises an exception rather than failing silently, so a clean run means the
webapp really is serving the new cards.

In [ ]:
from datetime import datetime
from pathlib import Path
import json

from openhexa.sdk import workspace
from openhexa.sdk.client import openhexa as hexa_client

In [ ]:
CARDS_DIR = (
    Path(workspace.files_path) / "pipelines/create_pipeline_cards/pipeline_cards"
)
WEBAPP_NAME = "SNT Pipelines Orchestrator - Cockpit"
WEBAPP_CARDS_PATH = "pipeline_cards.json"
HISTORICAL_DIR_NAME = "historical"


def log(message: str):
    """Print a timestamped log line, mirroring the pipeline's log style."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")


log(f"Workspace: {workspace.slug}")
log(f"Cards directory: {CARDS_DIR}")

In [ ]:
# --- Locate the pipeline_cards JSON ----------------------------------------
if not CARDS_DIR.is_dir():
    raise FileNotFoundError(
        f"Cards directory not found: {CARDS_DIR}. "
        "Has the `create_pipeline_cards` pipeline run in this workspace yet?"
    )

latest = CARDS_DIR / WEBAPP_CARDS_PATH
if not latest.is_file():
    raise FileNotFoundError(
        f"{WEBAPP_CARDS_PATH} not found in {CARDS_DIR}. "
        "Run the `create_pipeline_cards` pipeline first — it writes the file under this "
        "fixed name and archives the previous version into "
        f"'{HISTORICAL_DIR_NAME}/'."
    )

written = datetime.fromtimestamp(latest.stat().st_mtime)
age_hours = (datetime.now() - written).total_seconds() / 3600

log(f"Selected {latest.name}  (written {written:%Y-%m-%d %H:%M:%S}, {age_hours:.1f}h ago)")

# Previous versions the pipeline archived, newest first — context only.
historical_dir = CARDS_DIR / HISTORICAL_DIR_NAME
archived = sorted(historical_dir.glob("*.json"), reverse=True) if historical_dir.is_dir() else []
log(f"{len(archived)} archived version(s) in '{HISTORICAL_DIR_NAME}/'")
for other in archived[:3]:
    log(f"     older: {other.name}")

if age_hours > 24:
    log(
        f"WARNING: this file is {age_hours / 24:.1f} days old — re-run the pipeline if you expected fresher cards."
    )

In [ ]:
# --- Load and validate ------------------------------------------------------
content = latest.read_text(encoding="utf-8")
cards = json.loads(content)

pipelines = cards.get("pipelines", [])
if not pipelines:
    raise ValueError(
        f"{latest.name} contains no pipelines — refusing to deploy an empty catalog."
    )

log(
    f"Validated JSON: {len(pipelines)} pipelines, "
    f"workspace_slug='{cards.get('workspace_slug')}', "
    f"generated_at='{cards.get('generated_at')}', {len(content):,} bytes"
)

if cards.get("workspace_slug") != workspace.slug:
    log(
        f"WARNING: file was generated for workspace '{cards.get('workspace_slug')}' "
        f"but this notebook runs in '{workspace.slug}' — pipeline UUIDs are workspace-specific!"
    )

missing_uuid = [p.get("id") for p in pipelines if not p.get("uuid")]
if missing_uuid:
    log(
        f"WARNING: {len(missing_uuid)} pipeline(s) have no uuid: {', '.join(map(str, missing_uuid))}"
    )

In [ ]:
# --- Resolve the webapp by name + pre-flight the permission ----------------
QUERY_WEBAPPS = """
    query listWebapps($workspaceSlug: String!, $page: Int, $perPage: Int) {
        webapps(workspaceSlug: $workspaceSlug, page: $page, perPage: $perPage) {
            totalPages
            items { id name slug permissions { update } }
        }
    }
"""


def gql(query: str, variables: dict | None = None) -> dict:
    """Execute a GraphQL operation and return the `data` payload."""
    response = hexa_client.execute(query=query, variables=variables or {}).json()
    if "errors" in response:
        raise RuntimeError(f"GraphQL error: {response['errors']}")
    return response["data"]


webapps, page, total_pages = [], 1, 1
while page <= total_pages:
    result = gql(
        QUERY_WEBAPPS, {"workspaceSlug": workspace.slug, "page": page, "perPage": 50}
    )["webapps"]
    total_pages = result["totalPages"]
    webapps.extend(result["items"])
    page += 1

log(f"Found {len(webapps)} webapp(s) in '{workspace.slug}'")

matches = [w for w in webapps if w["name"] == WEBAPP_NAME]
if not matches:
    available = "\n  - ".join(sorted(w["name"] for w in webapps)) or "(none)"
    raise ValueError(f"No webapp named '{WEBAPP_NAME}'. Available:\n  - {available}")
if len(matches) > 1:
    raise ValueError(
        f"{len(matches)} webapps share the name '{WEBAPP_NAME}' — cannot disambiguate."
    )

webapp = matches[0]
log(f"Target webapp: '{webapp['name']}' (slug={webapp['slug']}, id={webapp['id']})")

can_update = (webapp.get("permissions") or {}).get("update")
if not can_update:
    raise PermissionError(
        f"This session cannot update '{webapp['name']}' (permissions.update = {can_update}). "
        "You need EDITOR or ADMIN on the workspace. Note that a pipeline-scoped token can "
        "NEVER pass this check — run this notebook as your own user."
    )
log("Pre-flight OK: this session has update permission on the webapp.")

In [ ]:
# --- Deploy ----------------------------------------------------------------
MUTATION_UPDATE_WEBAPP = """
    mutation updateWebapp($input: UpdateWebappInput!) {
        updateWebapp(input: $input) { success errors }
    }
"""

log(
    f"Deploying {latest.name} -> '{webapp['name']}':{WEBAPP_CARDS_PATH} ({len(content):,} bytes)"
)

result = gql(
    MUTATION_UPDATE_WEBAPP,
    {
        "input": {
            "id": webapp["id"],
            "files": [
                {"path": WEBAPP_CARDS_PATH, "content": content, "encoding": "TEXT"}
            ],
        }
    },
)["updateWebapp"]

if not result["success"]:
    raise RuntimeError(f"Deploy FAILED for '{webapp['name']}': {result['errors']}")

log(f"Deploy reported success for '{webapp['name']}'.")